In [3]:
import numpy as np
import remfile
import h5py
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import f_classif
import time
import warnings

# Suppress convergence and deprecation warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# ============================================================================
# Config
# ============================================================================

DANDISET_ID = "000402"
TARGET_STIMULI = ['Cinematic', 'Rendered', 'sports1m']  # Which stimuli to decode
INTERVAL_NAME = 'Clip'  # Which interval to use
N_TOP_NEURONS = 200  # How many neurons to select
N_FOLDS = 5  # Cross-validation folds

# Which ROI series to analyze
# Options: 'first', 'all', or specific list like ['RoiResponseSeries1', 'RoiResponseSeries3']
ROI_SERIES_TO_ANALYZE = 'all'

# Which files to process
# Options: 'all', or specific number like 3 files or smth
MAX_FILES = 3 

# ============================================================================
# Data loading functions
# ============================================================================

def get_all_nwb_assets(dandiset_id="000402"):
    """Get all NWB file assets from DANDI."""
    client = DandiAPIClient()
    dandiset = client.get_dandiset(dandiset_id, "draft")
    all_assets = list(dandiset.get_assets())
    nwb_assets = [a for a in all_assets if a.path.endswith('.nwb')]
    return nwb_assets


def load_nwb_file(asset):
    """Load an NWB file from a DANDI asset."""
    s3_url = asset.get_content_url(follow_redirects=1, strip_query=True)
    rf = remfile.File(s3_url)
    h5 = h5py.File(rf, "r")
    io = NWBHDF5IO(file=h5, load_namespaces=True)
    nwb = io.read()
    return nwb


def get_stimulus_timing(nwb, interval_name='Clip'):
    """Extract stimulus timing and labels from intervals."""
    interval = nwb.intervals[interval_name]
    starts = np.array(interval.start_time[:])
    stops = np.array(interval.stop_time[:])
    labels = np.array(interval.short_movie_name[:])
    return starts, stops, labels


def get_roi_series_list(nwb, roi_selection='first'):
    """
    Get list of ROI response series to analyze.
    
    Args:
        nwb: NWB file object
        roi_selection: 'first', 'all', or list of specific names
    
    Returns:
        List of (name, roi_response_series) tuples
    """
    ophys = nwb.processing['ophys']
    fluorescence = ophys.data_interfaces['Fluorescence']
    all_series = fluorescence.roi_response_series
    
    if roi_selection == 'first':
        first_name = list(all_series.keys())[0]
        return [(first_name, all_series[first_name])]
    elif roi_selection == 'all':
        return [(name, all_series[name]) for name in all_series.keys()]
    else:
        # Assume it's a list of specific names
        return [(name, all_series[name]) for name in roi_selection if name in all_series]


# ============================================================================
# Feature selection
# ============================================================================

def select_best_neurons(rs, timestamps, starts, stops, labels, 
                       target_stimuli, n_select=200):
    """Select most informative neurons using ANOVA F-test."""
    # Extract features for all neurons first
    all_neurons = list(range(rs.data.shape[1]))
    X_all, y_all = extract_neural_features(
        rs, timestamps, starts, stops, labels,
        all_neurons, target_stimuli
    )
    
    # Run F-test
    f_scores, _ = f_classif(X_all, y_all)
    best_indices = np.argsort(f_scores)[-n_select:]
    best_indices = sorted(best_indices.tolist())
    
    return best_indices


def extract_neural_features(rs, timestamps, starts, stops, labels,
                           neuron_indices, target_stimuli):
    """
    Extract mean neural activity for each stimulus presentation.
    
    Returns:
        X: Feature matrix (n_trials, n_neurons)
        y: Labels (n_trials,)
    """
    stim_to_idx = {stim: i for i, stim in enumerate(target_stimuli)}
    
    trials = []
    trial_labels = []
    
    for i in range(len(starts)):
        if labels[i] not in target_stimuli:
            continue
        
        # Find timepoints for this trial
        start_idx = np.searchsorted(timestamps, starts[i])
        stop_idx = np.searchsorted(timestamps, stops[i])
        
        # Extract neural activity and average over time
        neural_data = np.array(rs.data[start_idx:stop_idx, neuron_indices])
        mean_activity = np.mean(neural_data, axis=0)
        
        trials.append(mean_activity)
        trial_labels.append(stim_to_idx[labels[i]])
    
    X = np.array(trials)
    y = np.array(trial_labels)
    
    return X, y


# ============================================================================
# Decoding
# ============================================================================

def train_decoder(X, y, n_folds=5):
    """Train logistic regression decoder with grid search."""
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(random_state=42))
    ])
    
    # Updated param grid to avoid deprecated liblinear for multiclass
    param_grid = [
        {
            'classifier__C': [0.001, 0.01, 0.1, 1.0, 10, 100],
            'classifier__penalty': ['l2'],
            'classifier__solver': ['lbfgs', 'saga'],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [3000, 5000]
        },
        {
            'classifier__C': [0.001, 0.01, 0.1, 1.0, 10, 100],
            'classifier__penalty': ['l1'],
            'classifier__solver': ['saga'],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [3000, 5000]
        },
        {
            'classifier__penalty': [None],
            'classifier__solver': ['lbfgs', 'saga'],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__max_iter': [3000, 5000]
        }
    ]
    
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1,
        verbose=0,
        refit=True
    )
    
    grid_search.fit(X, y)
    
    return grid_search.best_estimator_, grid_search.best_params_


def evaluate_decoder(pipeline, X, y, n_folds=5):
    """Evaluate decoder with cross-validation."""
    from sklearn.base import clone
    
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_accuracies = []
    
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        model = clone(pipeline)
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        accuracy = np.mean(y_test == y_pred)
        fold_accuracies.append(accuracy)
    
    return fold_accuracies


# ============================================================================
# Pipeline
# ============================================================================

def analyze_single_roi(nwb, file_name, roi_name, roi_series, 
                      target_stimuli, n_neurons, n_folds):
    """Analyze a single ROI response series."""
    print(f"\n  Analyzing {roi_name}...")
    
    start_time = time.time()
    
    # Get stimulus timing
    starts, stops, labels = get_stimulus_timing(nwb, INTERVAL_NAME)
    timestamps = np.array(roi_series.timestamps[:])
    
    # Get dimensions
    n_total_neurons = roi_series.data.shape[1]
    n_timepoints = roi_series.data.shape[0]
    
    print(f"    Neurons: {n_total_neurons}, Timepoints: {n_timepoints}")
    
    # Select best neurons
    print(f"    Selecting top {n_neurons} neurons")
    neuron_indices = select_best_neurons(
        roi_series, timestamps, starts, stops, labels,
        target_stimuli, n_select=min(n_neurons, n_total_neurons)
    )
    
    # Extract features
    X, y = extract_neural_features(
        roi_series, timestamps, starts, stops, labels,
        neuron_indices, target_stimuli
    )
    
    best_pipeline, best_params = train_decoder(X, y, n_folds)
    fold_accs = evaluate_decoder(best_pipeline, X, y, n_folds)
    
    elapsed = time.time() - start_time
    
    result = {
        'file_name': file_name,
        'roi_name': roi_name,
        'n_total_neurons': n_total_neurons,
        'n_selected_neurons': len(neuron_indices),
        'n_trials': X.shape[0],
        'accuracy_mean': np.mean(fold_accs),
        'accuracy_std': np.std(fold_accs),
        'fold_accuracies': fold_accs,
        'best_params': best_params,
        'time_seconds': elapsed
    }
    
    print(f"    Accuracy: {result['accuracy_mean']*100:.2f}% +/- {result['accuracy_std']*100:.2f}%")
    print(f"    Time: {elapsed:.1f}s")
    
    return result


def run_full_pipeline():
    """Main pipeline to process all files and ROI series."""
    print("="*70)
    print("MULTI-FILE NEURAL DECODING PIPELINE")
    print("="*70)
    print(f"Target stimuli: {TARGET_STIMULI}")
    print(f"Interval: {INTERVAL_NAME}")
    print(f"Top neurons: {N_TOP_NEURONS}")
    print(f"ROI series: {ROI_SERIES_TO_ANALYZE}")
    print(f"Max files: {MAX_FILES if MAX_FILES else 'all'}")
    print()
    
    # Get all NWB files
    assets = get_all_nwb_assets(DANDISET_ID)
    
    if MAX_FILES:
        assets = assets[:MAX_FILES]
    
    print(f"Processing {len(assets)} files")
    print()
    
    # Store all results
    all_results = []
    
    # Process each file
    for file_idx, asset in enumerate(assets, 1):
        file_name = asset.path
        print(f"="*70)
        print(f"File {file_idx}/{len(assets)}: {file_name}")
        print(f"="*70)
        
        try:
            # Load NWB file
            nwb = load_nwb_file(asset)
            
            # Get ROI series to analyze
            roi_list = get_roi_series_list(nwb, ROI_SERIES_TO_ANALYZE)
            print(f"Found {len(roi_list)} ROI series to analyze")
            
            # Analyze each ROI series
            for roi_name, roi_series in roi_list:
                result = analyze_single_roi(
                    nwb, file_name, roi_name, roi_series,
                    TARGET_STIMULI, N_TOP_NEURONS, N_FOLDS
                )
                all_results.append(result)
        
        except Exception as e:
            print(f"ERROR processing {file_name}: {str(e)}")
            continue
    
    # Print summary
    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    print(f"Total analyses: {len(all_results)}")
    
    if all_results:
        # Sort by accuracy
        sorted_results = sorted(all_results, key=lambda x: x['accuracy_mean'], reverse=True)
        
        print("\nTop 10 by accuracy:")
        print(f"{'Rank':<6} {'File':<40} {'ROI':<20} {'Accuracy':<15}")
        print("-"*70)
        for i, res in enumerate(sorted_results[:10], 1):
            file_short = res['file_name'].split('/')[-1][:40]
            print(f"{i:<6} {file_short:<40} {res['roi_name']:<20} "
                  f"{res['accuracy_mean']*100:>5.2f}% +/- {res['accuracy_std']*100:>4.2f}%")
        
        # Overall stats
        all_accs = [r['accuracy_mean'] for r in all_results]
        print(f"\nOverall accuracy: {np.mean(all_accs)*100:.2f}% +/- {np.std(all_accs)*100:.2f}%")
        print(f"Best: {np.max(all_accs)*100:.2f}%")
        print(f"Worst: {np.min(all_accs)*100:.2f}%")
        
        total_time = sum(r['time_seconds'] for r in all_results)
        print(f"\nTotal time: {total_time/60:.1f} minutes")
    
    return all_results


# ============================================================================
# Running pipline
# ============================================================================

if __name__ == "__main__":
    results = run_full_pipeline()

MULTI-FILE NEURAL DECODING PIPELINE
Target stimuli: ['Cinematic', 'Rendered', 'sports1m']
Interval: Clip
Top neurons: 200
ROI series: all
Max files: 3

Processing 3 files

File 1/3: sub-17797/sub-17797_ses-4-scan-7_behavior+image+ophys.nwb
Found 8 ROI series to analyze

  Analyzing RoiResponseSeries1...
    Neurons: 643, Timepoints: 40000
    Selecting top 200 neurons
    Accuracy: 65.87% +/- 4.86%
    Time: 48.3s

  Analyzing RoiResponseSeries2...
    Neurons: 452, Timepoints: 40000
    Selecting top 200 neurons
    Accuracy: 64.82% +/- 5.99%
    Time: 38.6s

  Analyzing RoiResponseSeries3...
    Neurons: 1455, Timepoints: 40000
    Selecting top 200 neurons
    Accuracy: 71.86% +/- 4.87%
    Time: 79.3s

  Analyzing RoiResponseSeries4...
    Neurons: 1389, Timepoints: 40000
    Selecting top 200 neurons
    Accuracy: 72.62% +/- 8.42%
    Time: 70.2s

  Analyzing RoiResponseSeries5...
    Neurons: 1420, Timepoints: 40000
    Selecting top 200 neurons
    Accuracy: 72.12% +/- 4.63%
   